# Lesson 3: The Transformers Modelling Backend for vLLM

In Lessons 1 and 2, we saw that vLLM provides excellent inference performance. But there's a challenge: vLLM historically required a **hand-written model implementation** for every architecture it supports. With the Hugging Face model ecosystem growing rapidly, this doesn't scale.

The [**Transformers modelling backend**](https://docs.vllm.ai/en/latest/models/supported_models/#transformers) solves this by allowing vLLM to load and run any compatible Hugging Face model *without* a dedicated reimplementation. In this lesson, we'll understand:

1. The problem: why does vLLM need its own model implementations?
2. The solution: the Transformers backend architecture
3. How `Base.__init__` transforms a Hugging Face model for vLLM
4. The attention bridge: `vllm_flash_attention_forward`
5. Using the backend in practice

## The Problem: vLLM's Model Registry

vLLM maintains a registry that maps model architecture names (like `"Qwen3ForCausalLM"`) to vLLM-specific implementations. These implementations include optimizations like:

- **Tensor parallelism** - splitting layers across GPUs
- **PagedAttention kernels** - efficient KV cache management
- **Quantization support** - optimized kernels for quantized weights

However, there are advanced features like:

- **Expert Parallel Load Balancing (EPLB)** - compute balancing feature for MoE models
- **Eagle3** - speculative decoding method
- **LoRA**/**Multi-LoRA** - cheap fine-tunes/way to serve multiple fine-tunes simultaneously

which require specific additions to these implementations that new contributors will not know about.

Let's look at the registry:

In [ ]:
from vllm.model_executor.models.registry import _VLLM_MODELS, _TRANSFORMERS_BACKEND_MODELS

# Native vLLM implementations (hand-written)
print(f"Native vLLM model implementations: {len(_VLLM_MODELS) - len(_TRANSFORMERS_BACKEND_MODELS)}")
print("\nSome examples:")
native_models = {k: v for k, v in _VLLM_MODELS.items() if k not in _TRANSFORMERS_BACKEND_MODELS}
for name, (module, cls) in list(native_models.items())[:8]:
    print(f"  {name:40s} -> {module}.{cls}")

print(f"\n\nTransformers backend models: {len(_TRANSFORMERS_BACKEND_MODELS)}")
print("\nThese are the generic backend classes:")
for name, (module, cls) in _TRANSFORMERS_BACKEND_MODELS.items():
    print(f"  {name:50s} -> {module}.{cls}")

## The Transformers Backend Architecture

The backend uses a **mixin composition** pattern. Each mixin adds a specific capability **and** a vLLM interface/protocol class that tells vLLM what the model is capable of:

- `Base` — wraps a Hugging Face `PreTrainedModel` with vLLM optimizations. Implements `VllmModel`, `SupportsQuant`, `SupportsLoRA`, `SupportsPP`, `SupportsEagle`.
- `CausalMixin` — adds the language model head and logits processing. Implements `VllmModelForTextGeneration`.
- `MoEMixin` — adds MoE-specific weight loading and expert fusion. Implements `MixtureOfExperts`.
- `MultiModalMixin` — adds vision/multimodal support. Implements `SupportsMultiModal`, `SupportsMRoPE`.

These are composed into concrete classes:

```
TransformersForCausalLM         = CausalMixin + Base
TransformersMoEForCausalLM      = MoEMixin + CausalMixin + Base
TransformersMultiModalForCausalLM = MultiModalMixin + CausalMixin + Base
```

Qwen3 is a causal model, so it uses `TransformersForCausalLM`. The protocol classes are what vLLM checks at runtime to determine how to interact with the model — for example, whether it supports quantization, tensor parallelism, or expert load balancing.

In [ ]:
from vllm.model_executor.models.transformers import TransformersForCausalLM

print("TransformersForCausalLM MRO:")
for cls in TransformersForCausalLM.__mro__:
    print(f"  {cls.__name__}")

In [ ]:
from vllm.model_executor.models.transformers import TransformersMoEForCausalLM

print("TransformersMoEForCausalLM MRO:")
for cls in TransformersMoEForCausalLM.__mro__:
    print(f"  {cls.__name__}")

## How `Base.__init__` Works

The `Base` class is where the magic happens. When initialized, it takes a standard Hugging Face model and transforms it into something vLLM can use efficiently. Here are the key steps:

1. **`_patch_config()`** — sets the attention implementation to `"vllm"` so that Transformers dispatches attention calls to vLLM's attention kernels
2. **`_decorate_for_torch_compile()`** — wraps the decoder class with `@support_torch_compile` so vLLM knows it should use [`torch.compile`](https://pytorch.org/docs/stable/generated/torch.compile.html)*
3. **`AutoModel.from_config()`** — creates the Hugging Face model on a `"meta"` device (no actual memory allocated yet)
4. **`_create_hf_to_vllm_mapper()`** — builds a weight name mapper using information provided by Transformers so vLLM's weight loader can map the checkpoint weight names to the parameter names**
5. **`pipeline_parallel()`** — removes layers that don't belong on this pipeline parallel rank
6. **`recursive_replace()`** — swaps `nn.Linear` modules with vLLM's tensor-parallel variants, and `RMSNorm` with vLLM's optimized implementation
7. **`create_attention_instances()`** — creates vLLM `Attention` objects that manage the KV cache
8. **Replace input embeddings** — swaps `nn.Embedding` with `VocabParallelEmbedding`
9. **`init_parameters()`** — initializes any parameters that weren't replaced in previous steps

_*If a particular model isn't actually torch compilable, the user can always disable `torch.compile` via vLLM configuration._<br/>
_**Sometimes the Python implementation changes and Transformers uses these mappings so that checkpoints created with the old names still work._

Let's look at the actual source code:

In [ ]:
import inspect
from vllm.model_executor.models.transformers.base import Base

# Show the __init__ method
source = inspect.getsource(Base.__init__)
print(source)

## The MoE Mixin

For MoE models like OLMoE, the `MoEMixin` overrides `recursive_replace()` to run **before** `Base.recursive_replace()`. It walks the model tree looking for `experts` modules and replaces them with vLLM's `FusedMoE` — a single fused kernel that handles gating, top-k routing, and expert computation in one pass.

Note that since Transformers v5, expert weights are stored as packed 3D tensors (e.g. `gate_up_proj` of shape `(num_experts, 2 * intermediate_size, hidden_size)`) rather than as a `nn.ModuleList` with one module per expert. The mixin handles both formats.

There's an important subtlety in how routing works. vLLM's native MoE implementations bundle the router **inside** `FusedMoE` (e.g. `block_sparse_moe.experts` does both routing and expert computation). But in Transformers, the router sits **adjacent** to the experts — the MoE layer calls the router first, then passes top-k IDs and weights to the experts. Since the backend replaces only the `experts` module (not the whole MoE block), routing still happens on the Transformers side. The mixin handles this with a `custom_routing_function` that injects the routing decisions already made by Transformers rather than recomputing them inside `FusedMoE`.

This works, but it means the backend currently misses out on several optimizations that vLLM's native routing provides:

- **Fused top-k routing kernels** — fused softmax + top-k in a single kernel
- **Grouped top-k routing** — used by models like DeepSeek that group experts before top-k selection
- **Overlapped shared expert computation** — running shared experts concurrently with routed experts
- **`e_score_correction_bias`** — bias correction for expert scores

Future work aims to replace the entire `block_sparse_moe` module (not just `experts`) with `FusedMoE`, which would bring all of these optimizations to the Transformers backend.

Let's look at the key parts:

In [ ]:
import inspect
from vllm.model_executor.models.transformers.moe import MoEMixin, TransformersFusedMoE

# Show the MoEMixin.recursive_replace method
source = inspect.getsource(MoEMixin.recursive_replace)
print(source)

## The Attention Bridge

This is the key mechanism that makes the whole backend work.

Transformers has a global registry called `ALL_ATTENTION_FUNCTIONS` that maps attention implementation names to functions. The backend registers `vllm_flash_attention_forward` under the key `"vllm"`.

When a Transformers model calls attention, it looks up the implementation from `config._attn_implementation`. Since the backend set this to `"vllm"`, it dispatches to vLLM's attention function, which:

1. Looks up the `Attention` instance for this layer (from `attention_instances`)
2. Reshapes Q, K, V from Transformers format to vLLM format
3. Calls vLLM's optimized attention kernel with KV cache support

In [ ]:
import inspect
from vllm.model_executor.models.transformers.base import vllm_flash_attention_forward
from transformers.modeling_utils import ALL_ATTENTION_FUNCTIONS

# Show the registered attention functions
print("Registered attention functions in ALL_ATTENTION_FUNCTIONS:")
for name in ALL_ATTENTION_FUNCTIONS:
    print(f"  {name}")

print(f"\n'vllm' maps to: {ALL_ATTENTION_FUNCTIONS['vllm'].__name__}")

# Show the function source
print(f"\n{inspect.getsource(vllm_flash_attention_forward)}")

Notice how `attention_instances` is passed as a **kwarg** through the Transformers model's forward methods. This makes the `**kwargs` chain extremely important — if any layer drops `**kwargs`, the attention instances never reach the attention function, and inference breaks.

The flow looks like this:

```
Base.forward(...)
  -> kwargs = {"attention_instances": self.attention_instances, ...}
  -> HF Model.forward(**kwargs)            # kwargs contains attention_instances
    -> DecoderLayer.forward(**kwargs)      # passes it along
      -> Attention.forward(**kwargs)       # passes it along
        -> ALL_ATTENTION_FUNCTIONS["vllm"](**kwargs)  # uses it!
```

## Using the Backend

To use the [Transformers backend](https://docs.vllm.ai/en/latest/models/supported_models/#transformers), pass `model_impl="transformers"` when creating the `LLM`. This tells vLLM to use the generic backend instead of a model-specific implementation.

In [ ]:
import os
from vllm import LLM, SamplingParams

MODEL_NAME = "Qwen/Qwen3-0.6B"

os.environ["VLLM_CPU_KVCACHE_SPACE"] = "1"
llm = LLM(
    model=MODEL_NAME,
    max_model_len=4096,
    enforce_eager=True,
    model_impl="transformers",
)

sampling_params = SamplingParams(max_tokens=100, temperature=0.7)

outputs = llm.generate(
    ["Explain how mixture-of-experts models work:"],
    sampling_params,
)

print(outputs[0].outputs[0].text)

## What Makes a Model Compatible?

Not every Transformers model works with the backend out of the box. The model needs:

1. **`_supports_attention_backend = True`** on the `PreTrainedModel` class
2. **`ALL_ATTENTION_FUNCTIONS` dispatch** in the attention module's forward method
3. **`**kwargs` pass-through** at every level of the forward call chain
4. **`base_model_tp_plan`** in the config (for tensor parallelism)
5. **`base_model_pp_plan`** in the config (for pipeline parallelism)

In the next lesson, we'll reverse-engineer OLMoE to see exactly how each of these requirements is met in practice.

## Summary

The Transformers modelling backend enables vLLM to run **any compatible Hugging Face model** without a dedicated reimplementation. It works by:

1. Creating the Hugging Face model and **replacing** key components (linear layers, attention, embeddings) with vLLM-optimized versions
2. **Bridging attention** via `ALL_ATTENTION_FUNCTIONS["vllm"]`, which routes Transformers attention calls through vLLM's PagedAttention kernels
3. Passing `attention_instances` through the full `**kwargs` chain to reach the attention function

This means new models can work in vLLM as soon as they land in Transformers — no vLLM-specific code needed.